Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os

print(os.path.exists("/content/drive/MyDrive/train_embeddings.npy"))
print(os.path.exists("/content/drive/MyDrive/train_meta.json"))

True
True


Install independencies

In [4]:
!pip install "git+https://github.com/huggingface/transformers.git" accelerate bitsandbytes qwen-vl-utils --break-system-packages

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-n4tvyd4n
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-n4tvyd4n
  Resolved https://github.com/huggingface/transformers.git to commit a6ccf9354a4b9f6ec8d9c7482ed5e37734b6dbb2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 75.9 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11730936 sha256=47cb6cde983918a1f24fddf85970788dd89795b8ed3299d4a4964c761cfc944e
  Stored in directory: /tmp/pip-ephem-wheel-cache-hfl4tthj/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found ex

Load the model

In [5]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_name = "Qwen/Qwen3-VL-4B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading Qwen3-VL-4B...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)
model.eval()
print("Qwen3-VL-4B loaded!")

Loading Qwen3-VL-4B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Qwen3-VL-4B loaded!


For one image

For multi images zero shot

In [6]:
import os
import json
import re
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm

test_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
test_csv = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
output_json = "/content/drive/MyDrive/Qwen3VL4B_Misogyny_ZeroShot_pred.json"

test_df = pd.read_csv(test_csv)

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is misogynistic or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing both the image and the embedded text to determine whether it contains misogynistic content.
Step 2: If the meme contains any negative, insulting, stereotyping, or degrading reference to women, output Misogyny.
Step 3: If the meme does not contain any misogynistic content, output Non_Misogyny.

Output:
Your output should strictly follow the format:
Class labels: Misogyny or Non_Misogyny
Thought: Give your reason here"""

def parse_label(raw):
    m = re.search(r"Class labels?:\**\s*(Misogyny|Non_Misogyny)", raw, re.IGNORECASE)
    if m:
        return m.group(1)
    elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
        return "Non_Misogyny"
    elif "misogyn" in raw.lower():
        return "Misogyny"
    else:
        return "UNKNOWN"

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining_df = test_df[~test_df["filename"].isin(done_images)]
print(f"剩余待处理: {len(remaining_df)} 张")

for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="推理进度"):
    img_name = row["filename"]
    img_path = os.path.join(test_image_dir, img_name)

    if not os.path.exists(img_path):
        print(f"⚠️ 图片不存在: {img_name}")
        continue

    try:
        torch.cuda.empty_cache()
        image = Image.open(img_path).convert("RGB")
        if max(image.size) > 800:
            image.thumbnail((800, 800))

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)

        gen = outputs[0][inputs["input_ids"].shape[-1]:]
        raw = processor.decode(gen, skip_special_tokens=True).strip()
        label = parse_label(raw)

        # 统一大小写
        if label.lower() == "misogyny":
            label = "Misogyny"
        elif label.lower() == "non_misogyny":
            label = "Non_Misogyny"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw,
            "true_label": int(row["label"])
        })

        print(f"✅ {img_name} -> {label} (真实: {'Misogyny' if row['label']==1 else 'Non_Misogyny'})")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e),
            "true_label": int(row["label"])
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        torch.cuda.empty_cache()

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

发现已有成功结果 193 条，从断点继续...
剩余待处理: 147 张



推理进度:   1%|          | 1/147 [00:11<28:47, 11.83s/it]

✅ 937.jpg -> Non_Misogyny (真实: Non_Misogyny)



推理进度:   1%|▏         | 2/147 [00:19<22:31,  9.32s/it]

✅ 1632.jpg -> Non_Misogyny (真实: Non_Misogyny)


...

✅ 290.jpg -> Misogyny (真实: Misogyny)



推理进度:  99%|█████████▉| 146/147 [23:30<00:09,  9.61s/it]

✅ 1091.jpg -> Non_Misogyny (真实: Misogyny)



推理进度: 100%|██████████| 147/147 [23:43<00:00,  9.68s/it]

✅ 1103.jpg -> Non_Misogyny (真实: Non_Misogyny)

完成！共 340 条结果已保存
  Misogyny: 80
  Non_Misogyny: 260




Calculation of Zero shot

In [7]:
import json
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

output_json = "/content/drive/MyDrive/Qwen3VL4B_Misogyny_ZeroShot_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

def normalize_pred(x):
    x = str(x).strip().lower()
    if x == "misogyny":
        return "misogyny"
    return "non_misogyny"

def normalize_true(x):
    return "misogyny" if x == 1 else "non_misogyny"

y_true = [normalize_true(p["true_label"]) for p in predictions]
y_pred = [normalize_pred(p["predicted_label"]) for p in predictions]

print("总数:", len(predictions))
print("Unique TRUE labels:", sorted(set(y_true)))
print("Unique PRED labels:", sorted(set(y_pred)))

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

总数: 340
Unique TRUE labels: ['misogyny', 'non_misogyny']
Unique PRED labels: ['misogyny', 'non_misogyny']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.8000
MP   : 0.7740
MR   : 0.7322
MF1  : 0.7467
WP   : 0.7931
WR   : 0.8000
WF1  : 0.7918
-------------------------------------------------

Confusion Matrix:
[[ 58  46]
 [ 22 214]]

Classification Report:
              precision    recall  f1-score   support

    misogyny       0.72      0.56      0.63       104
non_misogyny       0.82      0.91      0.86       236

    accuracy                           0.80       340
   macro avg       0.77      0.73      0.75       340
weighted avg       0.79      0.80      0.79       340



Few shot with Rag

In [8]:
import os
import json
import re
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# ===============================
# 加载 CLIP + 训练集 embeddings
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

train_embeddings = np.load("/content/drive/MyDrive/misogyny_train_embeddings.npy")
with open("/content/drive/MyDrive/misogyny_train_meta.json", "r") as f:
    train_meta = json.load(f)

train_filenames = train_meta["filenames"]
train_labels = train_meta["labels"]
print(f"训练集 embeddings 加载完成，共 {len(train_embeddings)} 条")

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
test_csv = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
output_json = "/content/drive/MyDrive/Qwen3VL4B_Misogyny_FewShot_RAG_pred.json"

test_df = pd.read_csv(test_csv)

# ===============================
# RAG 检索函数
# ===============================
def get_clip_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        emb = outputs.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().numpy()

def retrieve_examples(test_emb):
    similarities = train_embeddings @ test_emb
    examples = {}
    for label in [0, 1]:
        label_indices = [i for i, l in enumerate(train_labels) if l == label]
        label_sims = [(i, similarities[i]) for i in label_indices]
        label_sims.sort(key=lambda x: x[1], reverse=True)
        examples[label] = label_sims[0][0]
    return examples

def build_prompt(example_indices):
    label_map = {0: "Non_Misogyny", 1: "Misogyny"}
    examples_text = ""
    for i, (label, idx) in enumerate(example_indices.items()):
        examples_text += f"Example {i+1}: Class label: {label_map[label]}\n"

    return f"""You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is misogynistic or not.

Below are 2 reference examples with their correct labels retrieved from similar memes:

{examples_text}
Now classify the following meme:

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing both the image and the embedded text, using the provided examples as reference to determine whether it contains misogynistic content.
Step 2: If the meme contains any negative, insulting, stereotyping, or degrading reference to women, output Misogyny.
Step 3: If the meme does not contain any misogynistic content, output Non_Misogyny.

Output:
Your output should strictly follow the format:
Class labels: Misogyny or Non_Misogyny
Thought: Give your reason here"""

def parse_label(raw):
    m = re.search(r"Class labels?:\**\s*(Misogyny|Non_Misogyny)", raw, re.IGNORECASE)
    if m:
        return m.group(1)
    elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
        return "Non_Misogyny"
    elif "misogyn" in raw.lower():
        return "Misogyny"
    else:
        return "UNKNOWN"

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining_df = test_df[~test_df["filename"].isin(done_images)]
print(f"剩余待处理: {len(remaining_df)} 张")

for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="推理进度"):
    img_name = row["filename"]
    img_path = os.path.join(test_image_dir, img_name)

    if not os.path.exists(img_path):
        print(f"⚠️ 图片不存在: {img_name}")
        continue

    try:
        torch.cuda.empty_cache()

        # RAG 检索
        test_emb = get_clip_embedding(img_path)
        example_indices = retrieve_examples(test_emb)
        prompt_text = build_prompt(example_indices)

        image = Image.open(img_path).convert("RGB")
        if max(image.size) > 800:
            image.thumbnail((800, 800))

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)

        gen = outputs[0][inputs["input_ids"].shape[-1]:]
        raw = processor.decode(gen, skip_special_tokens=True).strip()
        label = parse_label(raw)

        if label.lower() == "misogyny":
            label = "Misogyny"
        elif label.lower() == "non_misogyny":
            label = "Non_Misogyny"

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw,
            "true_label": int(row["label"])
        })

        print(f"✅ {img_name} -> {label} (真实: {'Misogyny' if row['label']==1 else 'Non_Misogyny'})")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e),
            "true_label": int(row["label"])
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        torch.cuda.empty_cache()

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

Loading CLIP...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

✅ 290.jpg -> Misogyny (真实: Misogyny)


推理进度: 100%|█████████▉| 339/340 [1:00:31<00:10, 10.35s/it]

✅ 1091.jpg -> Non_Misogyny (真实: Misogyny)


推理进度: 100%|██████████| 340/340 [1:00:45<00:00, 10.72s/it]

✅ 1103.jpg -> Non_Misogyny (真实: Non_Misogyny)

完成！共 340 条结果已保存
  Misogyny: 73
  Non_Misogyny: 267


Few shot Calculation

In [9]:
import json
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

output_json = "/content/drive/MyDrive/Qwen3VL4B_Misogyny_FewShot_RAG_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

def normalize_pred(x):
    x = str(x).strip().lower()
    if x == "misogyny":
        return "misogyny"
    return "non_misogyny"

def normalize_true(x):
    return "misogyny" if x == 1 else "non_misogyny"

y_true = [normalize_true(p["true_label"]) for p in predictions]
y_pred = [normalize_pred(p["predicted_label"]) for p in predictions]

print("总数:", len(predictions))
print("Unique TRUE labels:", sorted(set(y_true)))
print("Unique PRED labels:", sorted(set(y_pred)))

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

总数: 340
Unique TRUE labels: ['misogyny', 'non_misogyny']
Unique PRED labels: ['misogyny', 'non_misogyny']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.7971
MP   : 0.7762
MR   : 0.7194
MF1  : 0.7365
WP   : 0.7904
WR   : 0.7971
WF1  : 0.7855
-------------------------------------------------

Confusion Matrix:
[[ 54  50]
 [ 19 217]]

Classification Report:
              precision    recall  f1-score   support

    misogyny       0.74      0.52      0.61       104
non_misogyny       0.81      0.92      0.86       236

    accuracy                           0.80       340
   macro avg       0.78      0.72      0.74       340
weighted avg       0.79      0.80      0.79       340

